In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

df = pd.read_csv('../Dataset/processed/final_dataset_preprocessed_v2.csv')
y  = df['target'].values
n  = len(df)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)

for name, split in [('Train', y[:train_end]),
                    ('Val',   y[train_end:val_end]),
                    ('Test',  y[val_end:])]:
    c = Counter(split)
    print(f"{name:6s}: class0={c[0]:4d} ({c[0]/len(split):.1%})  "
          f"class1={c[1]:4d} ({c[1]/len(split):.1%})")


Train : class0=2900 (45.4%)  class1=3490 (54.6%)
Val   : class0= 608 (44.4%)  class1= 761 (55.6%)
Test  : class0= 271 (19.8%)  class1=1099 (80.2%)


This show test set is ~80% stressed — root cause of poor class 0 precision

In [2]:

def block_split(X, y, train_frac=0.70, val_frac=0.15, block_size=30):
    """
    Splits into blocks of `block_size` rows, then assigns each block
    to train/val/test proportionally — preserving within-block order.
    Ensures each split has a representative class distribution.
    """
    n_blocks = len(X) // block_size
    indices  = np.arange(n_blocks)
    np.random.seed(42)
    np.random.shuffle(indices)   # shuffle blocks, not rows

    n_train = int(n_blocks * train_frac)
    n_val   = int(n_blocks * val_frac)

    train_blocks = sorted(indices[:n_train])
    val_blocks   = sorted(indices[n_train:n_train + n_val])
    test_blocks  = sorted(indices[n_train + n_val:])

    def collect(block_ids):
        rows = []
        for b in block_ids:
            rows.extend(range(b * block_size, min((b+1) * block_size, len(X))))
        return np.array(rows)

    tr_idx  = collect(train_blocks)
    val_idx = collect(val_blocks)
    te_idx  = collect(test_blocks)

    return (X[tr_idx], y[tr_idx],
            X[val_idx], y[val_idx],
            X[te_idx],  y[te_idx])


FEATURE_COLS = [c for c in df.columns if c != 'target']
X_raw = df[FEATURE_COLS].values
y_raw = df['target'].values

# Log-compress wide-range cols first
for col in ['water_stress', 'combined_stress', 'temp_humidity_index']:
    idx = FEATURE_COLS.index(col)
    X_raw[:, idx] = np.log1p(X_raw[:, idx])

X_train, y_train, X_val, y_val, X_test, y_test = block_split(
    X_raw, y_raw, block_size=30
)

print("After block split:")
for name, split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    c = Counter(split)
    print(f"  {name:6s}: class0={c[0]:4d} ({c[0]/len(split):.1%})  "
          f"class1={c[1]:4d} ({c[1]/len(split):.1%})  n={len(split)}")

After block split:
  Train : class0=2575 (40.5%)  class1=3785 (59.5%)  n=6360
  Val   : class0= 541 (40.1%)  class1= 809 (59.9%)  n=1350
  Test  : class0= 663 (47.0%)  class1= 747 (53.0%)  n=1410


Standard 70/15/15 fails here because stress clusters at the end of the series. 

Fix: interleaved block split — preserves temporal order within each fold

but samples blocks from across the full timeline so each split sees both classes.

In [3]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
joblib.dump(scaler, 'scaler_v2.pkl')

print("Scaler fit on train blocks only")
print(f"Train mean: {X_train_sc.mean():.4f}  std: {X_train_sc.std():.4f}")

Scaler fit on train blocks only
Train mean: -0.0000  std: 1.0000


In [4]:
def make_sequences(X, y, time_steps=7):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i : i + time_steps])
        ys.append(y[i + time_steps])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

TIME_STEPS = 7

X_train_seq, y_train_seq = make_sequences(X_train_sc, y_train, TIME_STEPS)
X_val_seq,   y_val_seq   = make_sequences(X_val_sc,   y_val,   TIME_STEPS)
X_test_seq,  y_test_seq  = make_sequences(X_test_sc,  y_test,  TIME_STEPS)

print(f"Train sequences : {X_train_seq.shape}")
print(f"Val   sequences : {X_val_seq.shape}")
print(f"Test  sequences : {X_test_seq.shape}")

for name, yy in [('Train', y_train_seq), ('Val', y_val_seq), ('Test', y_test_seq)]:
    c = Counter(yy.astype(int))
    print(f"  {name}: class0={c[0]} ({c[0]/len(yy):.1%})  class1={c[1]} ({c[1]/len(yy):.1%})")

Train sequences : (6353, 7, 40)
Val   sequences : (1343, 7, 40)
Test  sequences : (1403, 7, 40)
  Train: class0=2568 (40.4%)  class1=3785 (59.6%)
  Val: class0=539 (40.1%)  class1=804 (59.9%)
  Test: class0=656 (46.8%)  class1=747 (53.2%)


In [6]:
# SMOTE on flattened sequences → reshape back
# Only applied to TRAIN — val and test stay untouched

from imblearn.over_sampling import SMOTE

n_seq, ts, n_feat = X_train_seq.shape
X_flat = X_train_seq.reshape(n_seq, ts * n_feat)   # flatten time × features

smote  = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_flat, y_train_seq.astype(int))

X_train_seq_bal = X_res.reshape(-1, ts, n_feat).astype(np.float32)
y_train_seq_bal = y_res.astype(np.float32)

print(f"Before SMOTE : {X_train_seq.shape}  |  {Counter(y_train_seq.astype(int))}")
print(f"After  SMOTE : {X_train_seq_bal.shape}  |  {Counter(y_train_seq_bal.astype(int))}")

Before SMOTE : (6353, 7, 40)  |  Counter({np.int64(1): 3785, np.int64(0): 2568})
After  SMOTE : (7570, 7, 40)  |  Counter({np.int64(0): 3785, np.int64(1): 3785})


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 64

class FarmDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(FarmDataset(X_train_seq_bal, y_train_seq_bal),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(FarmDataset(X_val_seq,   y_val_seq),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(FarmDataset(X_test_seq,  y_test_seq),
                          batch_size=BATCH_SIZE, shuffle=False)

xb, yb = next(iter(train_loader))
print(f"Batch shape — X: {xb.shape}  y: {yb.shape}  device: {DEVICE}")

d:\IoT_Precision_Agriculture\IOT_VENV\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Batch shape — X: torch.Size([64, 7, 40])  y: torch.Size([64])  device: cuda


In [8]:
class FarmLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(1)

INPUT_SIZE = X_train_seq_bal.shape[2]
model      = FarmLSTM(input_size=INPUT_SIZE).to(DEVICE)
print(f"Parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Parameters : 227,713


In [9]:
# After SMOTE training set is 50/50 — use plain BCE, no pos_weight
# Val/test are untouched real distributions

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)
print("Loss: BCEWithLogitsLoss (no pos_weight — SMOTE already balanced train set)")

Loss: BCEWithLogitsLoss (no pos_weight — SMOTE already balanced train set)


d:\IoT_Precision_Agriculture\IOT_VENV\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [10]:
from sklearn.metrics import roc_auc_score, f1_score

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)
        correct    += ((torch.sigmoid(logits) >= 0.5).long() == yb.long()).sum().item()
        total      += len(xb)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    probs_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb  = xb.to(DEVICE), yb.to(DEVICE)
            logits   = model(xb)
            loss     = criterion(logits, yb)
            total_loss += loss.item() * len(xb)
            probs    = torch.sigmoid(logits)
            correct += ((probs >= 0.5).long() == yb.long()).sum().item()
            total   += len(xb)
            probs_all.extend(probs.cpu().numpy())
            labels_all.extend(yb.cpu().numpy())
    auc = roc_auc_score(labels_all, probs_all)
    return total_loss / total, correct / total, auc

EPOCHS        = 50
best_val_loss = float('inf')
history       = {'train_loss':[], 'val_loss':[], 'train_acc':[],
                 'val_acc':[], 'val_auc':[]}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc             = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_auc  = eval_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_v2.pt')
        tag = '  ← saved'
    else:
        tag = ''

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | "
              f"tr_loss={tr_loss:.4f} acc={tr_acc:.4f} | "
              f"val_loss={val_loss:.4f} acc={val_acc:.4f} auc={val_auc:.4f}{tag}")

Epoch   1/50 | tr_loss=0.1832 acc=0.9324 | val_loss=0.2055 acc=0.9270 auc=0.9733  ← saved
Epoch   5/50 | tr_loss=0.0786 acc=0.9703 | val_loss=0.2016 acc=0.9322 auc=0.9793  ← saved
Epoch  10/50 | tr_loss=0.0747 acc=0.9721 | val_loss=0.1883 acc=0.9330 auc=0.9794
Epoch  15/50 | tr_loss=0.0615 acc=0.9750 | val_loss=0.1953 acc=0.9375 auc=0.9792
Epoch  20/50 | tr_loss=0.0489 acc=0.9804 | val_loss=0.2215 acc=0.9345 auc=0.9776
Epoch  25/50 | tr_loss=0.0399 acc=0.9836 | val_loss=0.2380 acc=0.9352 auc=0.9771
Epoch  30/50 | tr_loss=0.0344 acc=0.9852 | val_loss=0.2731 acc=0.9308 auc=0.9754
Epoch  35/50 | tr_loss=0.0284 acc=0.9872 | val_loss=0.2935 acc=0.9330 auc=0.9750
Epoch  40/50 | tr_loss=0.0264 acc=0.9878 | val_loss=0.3127 acc=0.9285 auc=0.9744
Epoch  45/50 | tr_loss=0.0237 acc=0.9900 | val_loss=0.3351 acc=0.9278 auc=0.9734
Epoch  50/50 | tr_loss=0.0235 acc=0.9893 | val_loss=0.3407 acc=0.9263 auc=0.9735


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

model.load_state_dict(torch.load('best_model_v2.pt', map_location=DEVICE))
model.eval()

probs_all, preds_all, labels_all = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(DEVICE))
        probs  = torch.sigmoid(logits).cpu().numpy()
        probs_all.extend(probs)
        preds_all.extend((probs >= 0.5).astype(int))
        labels_all.extend(yb.numpy().astype(int))

probs_all  = np.array(probs_all)
preds_all  = np.array(preds_all)
labels_all = np.array(labels_all)

print("=" * 52)
print("  BEFORE (old split, no SMOTE)")
print("=" * 52)
print("  Not stressed  prec=0.46  rec=0.76  f1=0.58") ## these valyes are from the previous version before block split + SMOTE 
print("  Stressed      prec=0.93  rec=0.78  f1=0.85")
print("  ROC-AUC : 0.8518")
print()
print("=" * 52)
print("  AFTER (block split + SMOTE)")
print("=" * 52)
print(classification_report(labels_all, preds_all,
                             target_names=['Not stressed', 'Stressed']))
print(f"ROC-AUC : {roc_auc_score(labels_all, probs_all):.4f}")

  BEFORE (old split, no SMOTE)
  Not stressed  prec=0.46  rec=0.76  f1=0.58
  Stressed      prec=0.93  rec=0.78  f1=0.85
  ROC-AUC : 0.8518

  AFTER (block split + SMOTE)
              precision    recall  f1-score   support

Not stressed       0.96      0.94      0.95       656
    Stressed       0.95      0.96      0.96       747

    accuracy                           0.95      1403
   macro avg       0.95      0.95      0.95      1403
weighted avg       0.95      0.95      0.95      1403

ROC-AUC : 0.9843


C:\Users\rajor\AppData\Local\Temp\ipykernel_11360\669793194.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_v2.pt', map_loca

In [12]:
# Find threshold that maximises macro F1 on val set
val_probs, val_labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb.to(DEVICE))
        val_probs.extend(torch.sigmoid(logits).cpu().numpy())
        val_labels.extend(yb.numpy())

val_probs  = np.array(val_probs)
val_labels = np.array(val_labels)

thresholds  = np.arange(0.20, 0.80, 0.01)
macro_f1s   = [f1_score(val_labels, (val_probs >= t).astype(int), average='macro')
               for t in thresholds]
best_thresh = thresholds[np.argmax(macro_f1s)]

print(f"Best threshold (macro F1 on val) : {best_thresh:.2f}")
print(f"Macro F1 at best threshold       : {max(macro_f1s):.4f}")
print(f"Macro F1 at 0.50                 : {f1_score(val_labels,(val_probs>=0.5).astype(int),average='macro'):.4f}")

print(f"\nTest at threshold={best_thresh:.2f}:")
print(classification_report(labels_all, (probs_all >= best_thresh).astype(int),
                             target_names=['Not stressed', 'Stressed']))

Best threshold (macro F1 on val) : 0.59
Macro F1 at best threshold       : 0.9393
Macro F1 at 0.50                 : 0.9367

Test at threshold=0.59:
              precision    recall  f1-score   support

Not stressed       0.94      0.96      0.95       656
    Stressed       0.97      0.95      0.96       747

    accuracy                           0.95      1403
   macro avg       0.95      0.95      0.95      1403
weighted avg       0.95      0.95      0.95      1403

